# Entraînement et test de différents modèles

### Préparation des données :

##### Import des librairies :

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, validation_curve, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier
from scipy.stats import randint, uniform, loguniform
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

##### Chargement des données d'entrainement et de test NON nettoyés car gradient boosting gère les NaN:

In [2]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

##### On traite les variables catégorielles pour qu'elles soient toutes des float/int :

In [3]:
ordre_age = {"<25": 25, "25-34": 30, "35-44": 40, "45-54": 50,
         "55-64": 60, "65-74": 70, ">74": 74}

data_train["applicant_age"] = data_train["applicant_age"].map(ordre_age)
data_train["co-applicant_age"] = data_train["co-applicant_age"].map(ordre_age)
data_test["applicant_age"] = data_test["applicant_age"].map(ordre_age)
data_test["co-applicant_age"] = data_test["co-applicant_age"].map(ordre_age)

In [4]:
ordre_debt = {"<20%": 20, "20%-<30%": 25, "30%-<36%": 33, "36": 36, "37": 37, "38": 38,
              "39": 39, "40": 40, "41": 41, "42": 42, "43": 43, "44": 44, "45": 45, "46": 46,
              "47": 47, "48": 48, "49": 49, "50%-60%": 55, ">60%": 60}

data_train["debt_to_income_ratio"] = data_train["debt_to_income_ratio"].map(ordre_debt)
data_test["debt_to_income_ratio"] = data_test["debt_to_income_ratio"].map(ordre_debt)

In [5]:
cols_nom = [
    "state_code", "conforming_loan_limit",
    "derived_loan_product_type", "derived_dwelling_category",
    "loan_type", "lien_status", "preapproval", "reverse_mortgage",
    "open-end_line_of_credit", "business_or_commercial_purpose",
    "negative_amortization", "interest_only_payment", "balloon_payment",
    "other_nonamortizing_features", "construction_method", "occupancy_type",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest", "total_units",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "derived_ethnicity", "derived_race", "derived_sex",
    "applicant_ethnicity-1", "co-applicant_ethnicity-1",
    "applicant_ethnicity_observed", "co-applicant_ethnicity_observed",
    "applicant_race-1", "co-applicant_race-1",
    "applicant_race_observed", "co-applicant_race_observed",
    "applicant_sex", "co-applicant_sex",
    "applicant_sex_observed", "co-applicant_sex_observed",
    "submission_of_application", "initially_payable_to_institution",
    "aus-1",
]

# Ordre alphabétique par défaut de OrdinalEncoder
enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

##### Ajout de variables ratios : 

In [6]:
def add_features(df):
    df = df.copy()
    inc = df["income"].where(df["income"] > 0) * 1000      # income est en milliers de $
    
    df["loan_to_income"]   = df["loan_amount"] / inc
    df["income_vs_msa"]    = inc / df["ffiec_msa_md_median_family_income"]
    df["loan_to_value_calc"] = df["loan_amount"] / df["property_value"]
    df["implied_value"]    = df["loan_amount"] / (df["loan_to_value_ratio"] / 100)
    df["value_gap"]        = df["implied_value"] / df["property_value"]
    df["monthly_principal"] = df["loan_amount"] / df["loan_term"]      # loan_term en mois
    df["pti_proxy"]        = df["monthly_principal"] / (inc / 12)      # proxy du payment-to-income
    df["debt_amount"]      = df["debt_to_income_ratio"] * inc / 100    # dette mensuelle/annuelle implicite
    df["residual_income"]  = inc - df["loan_amount"] / (df["loan_term"] / 12)
    
    df["n_missing"] = df.isna().sum(axis=1)
    return df.replace([np.inf, -np.inf], np.nan)

data_train_feat = add_features(data_train)
data_test_feat = add_features(data_test)

##### On sépare les covariables de la variable cible dans les données d'entraînement :

In [7]:
X_train = data_train_feat.drop(columns=["accepted", "id"])
Y_train = data_train_feat["accepted"]
X_test = data_test_feat.drop(columns=["id"])

## Modèles :

### Gradient boosting :

##### Entraînement :

Pas besoin de normaliser les données d'entraînement.

In [21]:
model_gdt_boost = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=1000,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=50,
    scoring="roc_auc",
    random_state=42,
)
model_gdt_boost.fit(X_train, Y_train)
print("Nombre d'arbres retenu :", model_gdt_boost.n_iter_)

Nombre d'arbres retenu : 297


##### Prédictions :

In [22]:
# Probabilité de la classe 1 (accepté)
y_proba = model_gdt_boost.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})

submission.to_csv("submission_gradient_boosting_try1_noclean_feat.csv", index=False)

En ne nettoyant pas les données, on obtient un score de 0.91523 sur Kaggle

En ne nettoyant pas les données et en ajoutant de nouvelles variables ratios, on obtient un score de 0.91740 sur Kaggle

##### Pour aller plus loin, on peut faire un réglage des hyperparamètres avec `RandomizedSearchCV` :

In [8]:
# Jeu de validation final, mis de côté
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, Y_train, test_size=0.2, stratify=Y_train, random_state=42
)

base = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=50,
    scoring="roc_auc",
    random_state=42,
)

param_dist = {
    "learning_rate":     loguniform(0.03, 0.2),
    "max_leaf_nodes":    randint(8, 64),         # au lieu de 15-256
    "min_samples_leaf":  randint(20, 200),       # au lieu de 20-500
    "l2_regularization": loguniform(1e-3, 10),
    "max_features":      [0.5, 0.7, 0.9, 1.0],
}

search = RandomizedSearchCV(
    base,
    param_distributions=param_dist,
    n_iter=30, cv=3, scoring="roc_auc",
    n_jobs=-1, random_state=42, verbose=2,
)
search.fit(X_tr, y_tr)

print("Meilleur AUC CV :", search.best_score_)
print("Meilleurs paramètres :", search.best_params_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


C:\Users\slese\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Le fichier spécifié est introuvable
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\slese\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\slese\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\slese\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^

Meilleur AUC CV : 0.9157377902802111
Meilleurs paramètres : {'l2_regularization': np.float64(0.002259279742015696), 'learning_rate': np.float64(0.04351041725942285), 'max_features': 0.5, 'max_leaf_nodes': 47, 'min_samples_leaf': 101}


No clean, Sans add_features :

Meilleur AUC CV : 0.9125886739289163
Meilleurs paramètres : {'l2_regularization': np.float64(1.050619945159704), 'learning_rate': np.float64(0.0672064407782783), 'max_features': 0.7, 'max_leaf_nodes': 49, 'min_samples_leaf': 23}

No clean, Avec add_features :

Meilleur AUC CV : 0.9157377902802111
Meilleurs paramètres : {'l2_regularization': np.float64(0.002259279742015696), 'learning_rate': np.float64(0.04351041725942285), 'max_features': 0.5, 'max_leaf_nodes': 47, 'min_samples_leaf': 101}

On réentraîne avec ces nouveaux hyperparamètres optimaux : 

In [9]:
model_gdt_boost_2 = HistGradientBoostingClassifier(
    max_iter=3000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=100,
    scoring="roc_auc",
    random_state=42,
    **search.best_params_,
)
model_gdt_boost_2.fit(X_train, Y_train)

print("Meilleurs paramètres :", search.best_params_)
print("Nb d'arbres retenu :", model_gdt_boost_2.n_iter_)

Meilleurs paramètres : {'l2_regularization': np.float64(0.002259279742015696), 'learning_rate': np.float64(0.04351041725942285), 'max_features': 0.5, 'max_leaf_nodes': 47, 'min_samples_leaf': 101}
Nb d'arbres retenu : 442


In [10]:
# Probabilité de la classe 1 (accepté)
y_proba = model_gdt_boost_2.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})

submission.to_csv("submission_gradient_boosting_try2_noclean_addfeat_reopt.csv", index=False)

Sur kaggle, on obtient un score de 0.91639 (en ne nettoyant pas les données et sans add_features)

Sur kaggle, on obtient un score de 0.91823 (en ne nettoyant pas les données et avec add_features)